# Harvard-only intercropping — one-click workbench

This notebook has exactly **one executable cell**. Run that cell once. It contains all imports, loading, feature construction, spatial evaluation, seven figures, and the final handoff. It imports no project modules, performs no network requests, and does not require a pipeline `COMPLETED.json` marker.


In [ ]:
from collections import OrderedDict
import hashlib
import json
import math
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pyproj import Transformer
from shapely import wkt as shapely_wkt
from shapely.geometry import box
from shapely.ops import transform as shapely_transform
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
).expanduser()
EXPORT_ROOT = Path(
    os.environ.get(
        "HARVARD_WORKBENCH_EXPORT_DIR",
        str(OUTPUT_DIR / "analysis" / "intercropping_harvard_only_workbench_v1"),
    )
).expanduser()

RANDOM_SEED = 24051995
BLOCK_METRES = 5_000
BOOTSTRAP_REPLICATES = int(os.environ.get("HARVARD_WORKBENCH_BOOTSTRAPS", "300"))
FROZEN_TESSERA_WINDOW = "w2"
TARGET_TRAIN_FPR = 0.10
SAFE_START = pd.Timestamp("2025-03-01")
SAFE_END = pd.Timestamp("2025-05-06")
FULL_END = pd.Timestamp("2025-07-01")
PHASES = OrderedDict(
    [
        ("mar", (pd.Timestamp("2025-03-01"), pd.Timestamp("2025-04-01"))),
        ("apr", (pd.Timestamp("2025-04-01"), SAFE_END)),
        ("may", (SAFE_END, pd.Timestamp("2025-06-01"))),
        ("jun", (pd.Timestamp("2025-06-01"), FULL_END)),
    ]
)
TARGET_LABELS = (
    "Maize",
    "Bean",
    "Irish Potato",
    "Rice",
    "Bean and Maize",
    "Irish Potato and Maize",
)
LABEL_COLORS = {
    "Maize": "#e69f00",
    "Bean": "#009e73",
    "Irish Potato": "#56b4e9",
    "Rice": "#7f7f7f",
    "Bean and Maize": "#d55e00",
    "Irish Potato and Maize": "#cc79a7",
}
LENS_COLORS = {
    "location_only": "#7f7f7f",
    "geometry_acquisition_only": "#a6a6a6",
    "nuisance_only": "#4d4d4d",
    "raw_safe": "#0072b2",
    "raw_safe_plus_context": "#009e73",
    "raw_full_retrospective": "#d55e00",
    "tessera_w2_frozen": "#cc79a7",
}
def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if not isinstance(value, (list, tuple, dict)):
        try:
            missing = pd.isna(value)
        except (TypeError, ValueError):
            missing = False
        if isinstance(missing, (bool, np.bool_)) and missing:
            return None
    return value


def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(_jsonable(payload), indent=2, sort_keys=True))
    temporary.replace(path)


def write_frame_atomic(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    frame.to_parquet(temporary, index=False)
    if pq.read_metadata(temporary).num_rows != len(frame):
        raise RuntimeError(f"row-count validation failed for {temporary}")
    temporary.replace(path)


def parquet_metadata(path):
    metadata = pq.read_metadata(path).metadata or {}
    return {
        key.decode("utf-8"): value.decode("utf-8")
        for key, value in metadata.items()
    }


def canonical_sha256(value):
    encoded = json.dumps(
        value, sort_keys=True, separators=(",", ":"), default=str
    ).encode()
    return hashlib.sha256(encoded).hexdigest()


def task_key_for(row):
    identity = {
        "epsg": int(row.utm_epsg),
        "work_x": int(row.work_x_index),
        "work_y": int(row.work_y_index),
        "chunk_x": int(row.chunk_x_index),
        "chunk_y": int(row.chunk_y_index),
    }
    return canonical_sha256(identity)[:24]




def require_inputs(root):
    required = [
        root / "run.json",
        root / "fields.parquet",
        root / "pixels.parquet",
        root / "field_pixels.parquet",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "The Harvard v2 static artifacts are required; missing: "
            + ", ".join(missing)
        )


def canonical_harvard_cohort(fields, memberships):
    fields = fields.copy()
    memberships = memberships.copy()
    fields["landcover"] = fields["landcover"].astype(str).str.strip()
    memberships["landcover"] = memberships["landcover"].astype(str).str.strip()
    if "source_id" not in fields:
        if "id" not in fields:
            raise KeyError("Harvard field source ID is unavailable")
        fields["source_id"] = fields["id"].astype(str)
    usable = fields[
        fields["pixel_count"].fillna(0).gt(0)
        & fields["geometry_sha256"].notna()
    ].copy()
    geometry_label_count = usable.groupby("geometry_sha256")["landcover"].nunique()
    conflicting_geometries = set(geometry_label_count[geometry_label_count.gt(1)].index)
    conflict_mask = usable["geometry_sha256"].isin(conflicting_geometries)
    nonconflicting = usable.loc[~conflict_mask].copy()
    canonical = (
        nonconflicting.sort_values(["geometry_sha256", "landcover", "field_uid"])
        .drop_duplicates(["geometry_sha256", "landcover"], keep="first")
        .loc[lambda frame: frame["landcover"].isin(TARGET_LABELS)]
        .reset_index(drop=True)
    )
    selected = memberships[memberships["field_uid"].isin(canonical["field_uid"])].copy()
    selected["canonical_pixel_field_count"] = selected.groupby("pixel_id")[
        "field_uid"
    ].transform("nunique")
    selected["canonical_pixel_label_count"] = selected.groupby("pixel_id")[
        "landcover"
    ].transform("nunique")
    selected = selected[
        selected["canonical_pixel_field_count"].eq(1)
        & selected["canonical_pixel_label_count"].eq(1)
    ].copy()
    fields_with_private_pixels = set(selected["field_uid"])
    canonical = canonical[canonical["field_uid"].isin(fields_with_private_pixels)].copy()
    selected = selected[selected["field_uid"].isin(canonical["field_uid"])].copy()
    selected = selected.drop_duplicates(["field_uid", "pixel_id"]).reset_index(drop=True)
    attrition = pd.DataFrame(
        [
            ("source_rows", len(fields)),
            ("positive_center_pixels", len(usable)),
            ("removed_label_conflict_geometry", int(conflict_mask.sum())),
            ("canonical_target_fields", len(canonical)),
            ("canonical_private_pixels", len(selected)),
        ],
        columns=["stage", "count"],
    )
    return canonical.reset_index(drop=True), selected, attrition


def _nanmedian(values, axis=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmedian(values, axis=axis)


def _naniqr(values, axis=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanpercentile(values, 75, axis=axis) - np.nanpercentile(
            values, 25, axis=axis
        )


def _ratio(numerator, denominator):
    output = np.full(numerator.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (np.abs(denominator) > 1e-6)
    output[valid] = numerator[valid] / denominator[valid]
    return output


def _phase_optical(payload, positions, start, end):
    dates = pd.to_datetime(payload["s2_days"], unit="D", origin="unix")
    take = np.flatnonzero((dates >= start) & (dates < end))
    pixel_count = len(positions)
    if not len(take):
        return {}, np.zeros(pixel_count, dtype=np.float32)
    values = payload["s2_values"][take][:, positions, :].astype(np.float32) / 10_000.0
    valid = payload["s2_valid"][take][:, positions].astype(bool)
    values[~valid, :] = np.nan
    red, green, nir = values[:, :, 0], values[:, :, 2], values[:, :, 3]
    narrow_nir, re1, re3 = values[:, :, 4], values[:, :, 5], values[:, :, 7]
    swir1, swir2 = values[:, :, 8], values[:, :, 9]
    variables = {
        "red": red,
        "green": green,
        "nir": nir,
        "narrow_nir": narrow_nir,
        "rededge1": re1,
        "rededge3": re3,
        "swir1": swir1,
        "swir2": swir2,
        "ndvi": _ratio(nir - red, nir + red),
        "ndre": _ratio(narrow_nir - re1, narrow_nir + re1),
        "ndmi": _ratio(nir - swir1, nir + swir1),
    }
    features = {f"s2_{name}_med": _nanmedian(array) for name, array in variables.items()}
    for name in ("ndvi", "ndre", "ndmi"):
        features[f"s2_{name}_tiqr"] = _naniqr(variables[name])
    return features, valid.sum(axis=0).astype(np.float32)


def _phase_radar(payload, positions, start, end):
    orbit_arrays = []
    orbit_valid = []
    for prefix in ("s1a", "s1d"):
        dates = pd.to_datetime(payload[f"{prefix}_days"], unit="D", origin="unix")
        take = np.flatnonzero((dates >= start) & (dates < end))
        if not len(take):
            continue
        values = payload[f"{prefix}_values"][take][:, positions, :].astype(np.float32)
        valid = payload[f"{prefix}_valid"][take][:, positions].astype(bool)
        band_valid = values != 0
        values = values / 200.0 - 50.0
        values[~band_valid] = np.nan
        values[~valid, :] = np.nan
        orbit_arrays.append(values)
        orbit_valid.append(valid)
    if not orbit_arrays:
        return {}, np.zeros(len(positions), dtype=np.float32)
    values = np.concatenate(orbit_arrays, axis=0)
    valid = np.concatenate(orbit_valid, axis=0)
    vv, vh = values[:, :, 0], values[:, :, 1]
    ratio = vh - vv
    return (
        {
            "s1_vv_db_med": _nanmedian(vv),
            "s1_vh_db_med": _nanmedian(vh),
            "s1_vh_minus_vv_med": _nanmedian(ratio),
            "s1_vh_minus_vv_tiqr": _naniqr(ratio),
        },
        valid.sum(axis=0).astype(np.float32),
    )


def pixel_phase_features(payload, positions):
    output = {}
    for phase, (start, end) in PHASES.items():
        optical, s2_count = _phase_optical(payload, positions, start, end)
        radar, s1_count = _phase_radar(payload, positions, start, end)
        output.update({f"{phase}_{name}": value for name, value in optical.items()})
        output.update({f"{phase}_{name}": value for name, value in radar.items()})
        output[f"{phase}_s2_obs"] = s2_count
        output[f"{phase}_s1_obs"] = s1_count
    return pd.DataFrame(output)


def _reference_shard(root, task_key):
    preferred = root / "embeddings" / f"window_id={FROZEN_TESSERA_WINDOW}" / f"{task_key}.parquet"
    if preferred.is_file():
        return preferred
    for window_id in ("w3", "w1", "w4"):
        candidate = root / "embeddings" / f"window_id={window_id}" / f"{task_key}.parquet"
        if candidate.is_file():
            return candidate
    return None


def extract_cached_features(root, memberships, run_fingerprint):
    group_columns = [
        "utm_epsg",
        "work_x_index",
        "work_y_index",
        "chunk_x_index",
        "chunk_y_index",
    ]
    pixel_parts = []
    embedding_parts = []
    audit_rows = []
    for group_key, selected in memberships.groupby(group_columns, sort=True):
        key = task_key_for(selected.iloc[0])
        reference = _reference_shard(root, key)
        if reference is None:
            audit_rows.append({"task_key": key, "status": "missing_reference_shard"})
            continue
        metadata = parquet_metadata(reference)
        if metadata.get("run_fingerprint") != run_fingerprint:
            raise RuntimeError(f"run fingerprint mismatch in {reference}")
        if metadata.get("task_key") != key:
            raise RuntimeError(f"task key mismatch in {reference}")
        fingerprint = metadata.get("task_fingerprint")
        if not fingerprint:
            raise RuntimeError(f"task fingerprint missing in {reference}")
        cache_path = root / "cache" / f"{fingerprint}.npz"
        if not cache_path.is_file():
            audit_rows.append({"task_key": key, "status": "missing_cache", "cache": str(cache_path)})
            continue
        with np.load(cache_path, allow_pickle=False) as raw:
            payload = {name: raw[name] for name in raw.files}
        cache_pixel_ids = [str(value) for value in payload["pixel_ids"]]
        position = {pixel_id: index for index, pixel_id in enumerate(cache_pixel_ids)}
        missing_pixels = sorted(set(selected["pixel_id"].astype(str)) - set(position))
        if missing_pixels:
            raise RuntimeError(f"{len(missing_pixels)} selected pixels are absent from {cache_path}")
        selected = selected.sort_values(["field_uid", "pixel_id"]).reset_index(drop=True)
        positions = np.asarray([position[value] for value in selected["pixel_id"].astype(str)], dtype=np.int64)
        features = pixel_phase_features(payload, positions)
        features.insert(0, "pixel_id", selected["pixel_id"].astype(str).to_numpy())
        features.insert(0, "field_uid", selected["field_uid"].astype(str).to_numpy())
        pixel_parts.append(features)

        frozen_path = root / "embeddings" / f"window_id={FROZEN_TESSERA_WINDOW}" / f"{key}.parquet"
        frozen_complete = 0
        if frozen_path.is_file():
            shard = pd.read_parquet(
                frozen_path,
                columns=["field_uid", "pixel_id", "outcome", "embedding"],
            )
            shard = shard.merge(
                selected[["field_uid", "pixel_id"]],
                on=["field_uid", "pixel_id"],
                how="inner",
                validate="one_to_one",
            )
            shard = shard[shard["outcome"].eq("complete") & shard["embedding"].notna()]
            if len(shard):
                matrix = np.vstack(shard["embedding"].map(lambda value: np.asarray(value, dtype=np.float32)))
                if matrix.shape[1] != 128:
                    raise RuntimeError(f"unexpected embedding width in {frozen_path}: {matrix.shape}")
                embedded = pd.DataFrame(matrix, columns=[f"tessera_{index:03d}" for index in range(128)])
                embedded.insert(0, "pixel_id", shard["pixel_id"].astype(str).to_numpy())
                embedded.insert(0, "field_uid", shard["field_uid"].astype(str).to_numpy())
                embedding_parts.append(embedded)
                frozen_complete = len(embedded)

        def date_span(days):
            if not len(days):
                return (None, None)
            dates = pd.to_datetime(days, unit="D", origin="unix")
            return (dates.min(), dates.max())

        s2_min, s2_max = date_span(payload["s2_days"])
        s1_days = np.concatenate([payload["s1a_days"], payload["s1d_days"]])
        s1_min, s1_max = date_span(s1_days)
        audit_rows.append(
            {
                "task_key": key,
                "status": "complete",
                "selected_fields": selected["field_uid"].nunique(),
                "selected_pixels": len(selected),
                "cache_pixels": len(cache_pixel_ids),
                "cache_bytes": cache_path.stat().st_size,
                "s2_first": s2_min,
                "s2_last": s2_max,
                "s1_first": s1_min,
                "s1_last": s1_max,
                "frozen_w2_complete_pixels": frozen_complete,
                "cache_path": str(cache_path),
            }
        )
    if not pixel_parts:
        raise RuntimeError("No Harvard raw-cache pixels could be recovered")
    return (
        pd.concat(pixel_parts, ignore_index=True),
        pd.concat(embedding_parts, ignore_index=True) if embedding_parts else pd.DataFrame(),
        pd.DataFrame(audit_rows),
    )


def aggregate_pixel_features(pixel_features):
    feature_columns = [
        column for column in pixel_features.columns if column not in {"field_uid", "pixel_id"}
    ]
    grouped = pixel_features.groupby("field_uid", sort=True)[feature_columns]
    medians = grouped.median().add_suffix("__fmed")
    q75 = grouped.quantile(0.75)
    q25 = grouped.quantile(0.25)
    iqrs = (q75 - q25).add_suffix("__fiqr")
    result = medians.join(iqrs).reset_index()
    contrasts = {}
    for late, early in (("apr", "mar"), ("may", "apr"), ("jun", "may")):
        late_columns = [column for column in result.columns if column.startswith(f"{late}_")]
        for late_column in late_columns:
            early_column = f"{early}_{late_column[len(late) + 1:]}"
            if early_column in result:
                name = f"delta_{late}_{early}__{late_column[len(late) + 1:]}"
                contrasts[name] = result[late_column] - result[early_column]
    if contrasts:
        result = pd.concat([result, pd.DataFrame(contrasts, index=result.index)], axis=1)
    return result


def aggregate_embeddings(embedding_pixels):
    if embedding_pixels.empty:
        return pd.DataFrame(columns=["field_uid", "tessera_pixel_count"])
    columns = [column for column in embedding_pixels if column.startswith("tessera_")]
    grouped = embedding_pixels.groupby("field_uid", sort=True)[columns]
    means = grouped.mean().add_suffix("__mean")
    stds = grouped.std(ddof=0).add_suffix("__std")
    counts = grouped.size().rename("tessera_pixel_count")
    return means.join(stds).join(counts).reset_index()


def geometry_context(fields, memberships):
    rows = []
    membership_groups = {
        field_uid: group for field_uid, group in memberships.groupby("field_uid", sort=False)
    }
    for field in fields.itertuples(index=False):
        field_uid = str(field.field_uid)
        pixels = membership_groups.get(field_uid)
        if pixels is None or pixels.empty:
            continue
        epsg = int(field.utm_epsg)
        geometry = shapely_wkt.loads(str(field.wkt))
        if not geometry.is_valid:
            geometry = geometry.buffer(0)
        representative = geometry.representative_point()
        longitude = float(representative.x)
        latitude = float(representative.y)
        transformer = Transformer.from_crs(4326, epsg, always_xy=True)
        x, y = transformer.transform(longitude, latitude)
        projected = shapely_transform(transformer.transform, geometry)
        coverage = []
        for pixel in pixels.itertuples(index=False):
            cell = box(
                int(pixel.pixel_x_index) * 10.0,
                int(pixel.pixel_y_index) * 10.0,
                (int(pixel.pixel_x_index) + 1) * 10.0,
                (int(pixel.pixel_y_index) + 1) * 10.0,
            )
            coverage.append(float(projected.intersection(cell).area / 100.0))
        coverage_array = np.asarray(coverage, dtype=float)
        pixel_count = len(pixels)
        rows.append(
            {
                "field_uid": field_uid,
                "longitude": longitude,
                "latitude": latitude,
                "projected_x_km": x / 1_000.0,
                "projected_y_km": y / 1_000.0,
                "spatial_block": f"{epsg}:{math.floor(x / BLOCK_METRES)}:{math.floor(y / BLOCK_METRES)}",
                "private_pixel_count": pixel_count,
                "log_area_m2": np.log1p(float(field.area_m2)),
                "log_private_pixel_count": np.log1p(pixel_count),
                "median_footprint_coverage": float(np.nanmedian(coverage_array)),
                "minimum_footprint_coverage": float(np.nanmin(coverage_array)),
                "edge_pixel_fraction": float(np.mean(coverage_array < 0.5)),
                "resolution_stratum": (
                    "1-2 pixels" if pixel_count < 3 else "3-9 pixels" if pixel_count < 10 else "10+ pixels"
                ),
            }
        )
    return pd.DataFrame(rows)


def crop_targets(frame):
    output = frame.copy()
    label = output["landcover"].astype(str)
    output["is_intercrop"] = label.str.contains(" and ", regex=False).astype(int)
    output["has_maize"] = label.str.contains("Maize", regex=False).astype(int)
    output["has_bean"] = label.str.contains("Bean", regex=False).astype(int)
    output["has_potato"] = label.str.contains("Irish Potato", regex=False).astype(int)
    output["has_rice"] = label.eq("Rice").astype(int)
    return output


def make_feature_sets(frame):
    location = ["longitude", "latitude", "projected_x_km", "projected_y_km"]
    geometry = [
        "log_area_m2",
        "log_private_pixel_count",
        "median_footprint_coverage",
        "minimum_footprint_coverage",
        "edge_pixel_fraction",
    ]
    safe_obs = [
        column
        for column in frame
        if (column.startswith("mar_") or column.startswith("apr_"))
        and "_obs__" in column
    ]
    safe_raw = [
        column
        for column in frame
        if (
            column.startswith("mar_")
            or column.startswith("apr_")
            or column.startswith("delta_apr_mar__")
        )
        and "_obs__" not in column
    ]
    full_raw = [
        column
        for column in frame
        if (
            column.startswith("mar_")
            or column.startswith("apr_")
            or column.startswith("may_")
            or column.startswith("jun_")
            or column.startswith("delta_")
        )
        and "_obs__" not in column
    ]
    tessera = [column for column in frame if column.startswith("tessera_") and column != "tessera_pixel_count"]
    feature_sets = OrderedDict(
        [
            ("location_only", location),
            ("geometry_acquisition_only", geometry + safe_obs),
            ("nuisance_only", location + geometry + safe_obs),
            ("raw_safe", safe_raw),
            ("raw_safe_plus_context", safe_raw + location + geometry + safe_obs),
            ("raw_full_retrospective", full_raw),
            ("tessera_w2_frozen", tessera),
        ]
    )
    return OrderedDict(
        (name, [column for column in columns if column in frame])
        for name, columns in feature_sets.items()
    )


TASK_SPECS = OrderedDict(
    [
        (
            "generic_intercrop",
            {
                "category": "intercropping",
                "labels": TARGET_LABELS,
                "target": "is_intercrop",
                "scope": "exploratory_primary",
            },
        ),
        (
            "bean_maize_mixture",
            {
                "category": "intercropping",
                "labels": ("Bean", "Maize", "Bean and Maize"),
                "target": "is_intercrop",
                "scope": "exploratory_primary",
            },
        ),
        (
            "potato_maize_mixture",
            {
                "category": "intercropping",
                "labels": ("Irish Potato", "Maize", "Irish Potato and Maize"),
                "target": "is_intercrop",
                "scope": "descriptive_small_n",
            },
        ),
        (
            "maize_presence",
            {"category": "crop_presence", "labels": TARGET_LABELS, "target": "has_maize", "scope": "exploratory"},
        ),
        (
            "bean_presence",
            {"category": "crop_presence", "labels": TARGET_LABELS, "target": "has_bean", "scope": "exploratory"},
        ),
        (
            "potato_presence",
            {"category": "crop_presence", "labels": TARGET_LABELS, "target": "has_potato", "scope": "exploratory"},
        ),
        (
            "rice_presence",
            {"category": "crop_presence", "labels": TARGET_LABELS, "target": "has_rice", "scope": "exploratory"},
        ),
    ]
)


def spatial_folds(frame, target, maximum=5):
    y = frame[target].to_numpy(dtype=int)
    groups = frame["spatial_block"].astype(str).to_numpy()
    if len(np.unique(y)) < 2:
        return [], "one_class"
    unique_groups = np.unique(groups)
    for split_count in range(min(maximum, len(unique_groups)), 1, -1):
        splitter = StratifiedGroupKFold(
            n_splits=split_count, shuffle=True, random_state=RANDOM_SEED
        )
        try:
            folds = list(splitter.split(np.zeros(len(frame)), y, groups))
        except ValueError:
            continue
        valid = all(
            len(np.unique(y[train])) == 2
            and len(np.unique(y[test])) == 2
            and not (set(groups[train]) & set(groups[test]))
            for train, test in folds
        )
        if valid:
            return folds, f"{split_count}_fold_spatial"
    return [], "not_spatially_testable"


def calibration_split(frame, train_indices, target):
    training = frame.iloc[train_indices].reset_index(drop=True)
    folds, status = spatial_folds(training, target, maximum=3)
    if not folds:
        return None, None, status
    fit_local, calibration_local = folds[0]
    return train_indices[fit_local], train_indices[calibration_local], status


def model_pipeline():
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=3_000,
                    random_state=RANDOM_SEED,
                    solver="liblinear",
                ),
            ),
        ]
    )


def _usable_columns(frame, train_indices, columns):
    usable = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
        if np.isfinite(values[train_indices]).sum() >= 2:
            usable.append(column)
    return usable


def _higher_quantile(values, quantile):
    try:
        return float(np.quantile(values, quantile, method="higher"))
    except TypeError:
        return float(np.quantile(values, quantile, interpolation="higher"))


def evaluate_lens(frame, target, folds, lens, columns, task_name, spec):
    rows = []
    y = frame[target].to_numpy(dtype=int)
    for fold_id, (train, test) in enumerate(folds):
        usable = _usable_columns(frame, train, columns)
        if not usable:
            continue
        X = frame[usable].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
        primary_model = model_pipeline()
        primary_model.fit(X[train], y[train])
        primary_score = primary_model.predict_proba(X[test])[:, 1]

        fit, calibration, calibration_status = calibration_split(frame, train, target)
        operating_score = np.full(len(test), np.nan, dtype=float)
        threshold = np.nan
        decision = np.full(len(test), np.nan, dtype=float)
        calibration_fpr = np.nan
        operating_feature_count = 0
        if fit is not None and calibration is not None:
            operating_columns = _usable_columns(frame, fit, usable)
            operating_feature_count = len(operating_columns)
            if operating_columns:
                operating_X = frame[operating_columns].apply(
                    pd.to_numeric, errors="coerce"
                ).to_numpy(dtype=float)
                calibration_model = model_pipeline()
                calibration_model.fit(operating_X[fit], y[fit])
                calibration_score = calibration_model.predict_proba(
                    operating_X[calibration]
                )[:, 1]
                negative_scores = calibration_score[y[calibration] == 0]
                if len(negative_scores):
                    threshold = _higher_quantile(
                        negative_scores, 1.0 - TARGET_TRAIN_FPR
                    )
                    if np.mean(negative_scores >= threshold) > TARGET_TRAIN_FPR:
                        threshold = float(np.nextafter(threshold, np.inf))
                    calibration_fpr = float(np.mean(negative_scores >= threshold))
                    operating_score = calibration_model.predict_proba(operating_X[test])[:, 1]
                    decision = (operating_score >= threshold).astype(float)
            else:
                calibration_status = "no_inner_training_features"

        for position, row_index in enumerate(test):
            source = frame.iloc[row_index]
            rows.append(
                {
                    "task": task_name,
                    "task_category": spec["category"],
                    "scope": spec["scope"],
                    "lens": lens,
                    "fold": fold_id,
                    "field_uid": source["field_uid"],
                    "landcover": source["landcover"],
                    "spatial_block": source["spatial_block"],
                    "resolution_stratum": source["resolution_stratum"],
                    "y_true": int(y[row_index]),
                    "score": float(primary_score[position]),
                    "operating_score": float(operating_score[position]),
                    "threshold": float(threshold),
                    "decision": float(decision[position]),
                    "n_features": len(usable),
                    "operating_n_features": operating_feature_count,
                    "calibration_false_positive_rate": calibration_fpr,
                    "outer_train_fields": len(train),
                    "outer_test_fields": len(test),
                    "calibration_status": calibration_status,
                }
            )
    return rows


def binary_metrics(frame):
    valid = frame[np.isfinite(frame["score"])].copy()
    if valid.empty or valid["y_true"].nunique() < 2:
        return {
            "n": len(valid),
            "positive_n": int(valid["y_true"].sum()) if len(valid) else 0,
            "prevalence": float(valid["y_true"].mean()) if len(valid) else np.nan,
            "auroc": np.nan,
            "average_precision": np.nan,
            "ap_lift": np.nan,
            "recall": np.nan,
            "false_positive_rate": np.nan,
            "operating_n": 0,
        }
    y = valid["y_true"].to_numpy(dtype=int)
    score = valid["score"].to_numpy(dtype=float)
    prevalence = float(np.mean(y))
    operating = valid[valid["decision"].notna()].copy()
    recall = np.nan
    fpr = np.nan
    if not operating.empty and operating["y_true"].nunique() == 2:
        matrix = confusion_matrix(
            operating["y_true"].astype(int), operating["decision"].astype(int), labels=[0, 1]
        )
        tn, fp, fn, tp = matrix.ravel()
        recall = tp / (tp + fn) if tp + fn else np.nan
        fpr = fp / (fp + tn) if fp + tn else np.nan
    ap = float(average_precision_score(y, score))
    return {
        "n": len(valid),
        "positive_n": int(y.sum()),
        "prevalence": prevalence,
        "auroc": float(roc_auc_score(y, score)),
        "average_precision": ap,
        "ap_lift": ap / prevalence if prevalence else np.nan,
        "recall": recall,
        "false_positive_rate": fpr,
        "operating_n": len(operating),
    }


def bootstrap_interval(frame, metric_name, seed_offset=0):
    groups = frame["spatial_block"].dropna().astype(str).unique()
    if len(groups) < 2:
        return (np.nan, np.nan)
    grouped = {group: frame[frame["spatial_block"].astype(str).eq(group)] for group in groups}
    rng = np.random.default_rng(RANDOM_SEED + seed_offset)
    estimates = []
    for _ in range(BOOTSTRAP_REPLICATES):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        replicate = pd.concat([grouped[group] for group in sampled], ignore_index=True)
        value = binary_metrics(replicate).get(metric_name, np.nan)
        if np.isfinite(value):
            estimates.append(float(value))
    if len(estimates) < max(30, BOOTSTRAP_REPLICATES // 5):
        return (np.nan, np.nan)
    return tuple(np.quantile(estimates, [0.025, 0.975]).astype(float))


def summarize_performance(predictions):
    rows = []
    for index, ((task, lens), group) in enumerate(predictions.groupby(["task", "lens"], sort=False)):
        metrics = binary_metrics(group)
        auroc_low, auroc_high = bootstrap_interval(group, "auroc", seed_offset=index * 17)
        ap_low, ap_high = bootstrap_interval(group, "average_precision", seed_offset=index * 17 + 1)
        rows.append(
            {
                "task": task,
                "lens": lens,
                "task_category": group["task_category"].iloc[0],
                "scope": group["scope"].iloc[0],
                **metrics,
                "auroc_ci_low": auroc_low,
                "auroc_ci_high": auroc_high,
                "ap_ci_low": ap_low,
                "ap_ci_high": ap_high,
            }
        )
    return pd.DataFrame(rows)


def paired_delta(predictions, task, lens_a, lens_b, metric="auroc"):
    left = predictions[(predictions["task"].eq(task)) & (predictions["lens"].eq(lens_a))]
    right = predictions[(predictions["task"].eq(task)) & (predictions["lens"].eq(lens_b))]
    paired = left.merge(
        right[["field_uid", "score"]].rename(columns={"score": "score_b"}),
        on="field_uid",
        how="inner",
        validate="one_to_one",
    )
    if paired.empty or paired["y_true"].nunique() < 2:
        return {"n": len(paired), "delta": np.nan, "ci_low": np.nan, "ci_high": np.nan}

    def value(frame, column):
        if frame["y_true"].nunique() < 2:
            return np.nan
        if metric == "auroc":
            return roc_auc_score(frame["y_true"], frame[column])
        return average_precision_score(frame["y_true"], frame[column])

    observed = float(value(paired, "score") - value(paired, "score_b"))
    groups = paired["spatial_block"].astype(str).unique()
    grouped = {group: paired[paired["spatial_block"].astype(str).eq(group)] for group in groups}
    rng = np.random.default_rng(RANDOM_SEED + int(hashlib.sha256(f"{task}:{lens_a}:{lens_b}:{metric}".encode()).hexdigest()[:8], 16))
    estimates = []
    for _ in range(BOOTSTRAP_REPLICATES):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        replicate = pd.concat([grouped[group] for group in sampled], ignore_index=True)
        estimate = value(replicate, "score") - value(replicate, "score_b")
        if np.isfinite(estimate):
            estimates.append(float(estimate))
    low, high = (np.nan, np.nan)
    if len(estimates) >= max(30, BOOTSTRAP_REPLICATES // 5):
        low, high = np.quantile(estimates, [0.025, 0.975])
    return {"n": len(paired), "delta": observed, "ci_low": low, "ci_high": high}


def summarize_resolution(predictions):
    rows = []
    selected = predictions[predictions["lens"].eq("raw_safe")]
    for (task, stratum), group in selected.groupby(["task", "resolution_stratum"], sort=False):
        rows.append({"task": task, "resolution_stratum": stratum, **binary_metrics(group)})
    columns = [
        "task",
        "resolution_stratum",
        "n",
        "positive_n",
        "prevalence",
        "auroc",
        "average_precision",
        "ap_lift",
        "recall",
        "false_positive_rate",
        "operating_n",
    ]
    return pd.DataFrame(rows, columns=columns)


def metric_value(performance, task, lens, metric):
    match = performance[(performance["task"].eq(task)) & (performance["lens"].eq(lens))]
    return float(match.iloc[0][metric]) if len(match) else np.nan


def build_scorecard(performance, deltas, resolution):
    bean_raw = metric_value(performance, "bean_maize_mixture", "raw_safe", "auroc")
    bean_nuisance = metric_value(performance, "bean_maize_mixture", "nuisance_only", "auroc")
    bean_full = metric_value(performance, "bean_maize_mixture", "raw_full_retrospective", "auroc")
    presence = performance[
        performance["task_category"].eq("crop_presence") & performance["lens"].eq("raw_safe")
    ]
    best_presence = float(presence["auroc"].max()) if len(presence) else np.nan
    bean_resolution = resolution[resolution["task"].eq("bean_maize_mixture")]
    large = bean_resolution[bean_resolution["resolution_stratum"].eq("10+ pixels")]
    large_auroc = float(large["auroc"].iloc[0]) if len(large) else np.nan
    rows = [
        {
            "hypothesis": "post-survey timing matters",
            "diagnostic": "retrospective minus safe Bean-Maize AUROC",
            "value": bean_full - bean_raw,
            "flag": bool(np.isfinite(bean_full - bean_raw) and bean_full - bean_raw >= 0.05),
        },
        {
            "hypothesis": "geography/geometry carries the label",
            "diagnostic": "safe raw minus nuisance Bean-Maize AUROC",
            "value": bean_raw - bean_nuisance,
            "flag": bool(np.isfinite(bean_raw - bean_nuisance) and bean_raw - bean_nuisance <= 0.03),
        },
        {
            "hypothesis": "10 m support is limiting",
            "diagnostic": "10+ pixel minus all-field Bean-Maize AUROC",
            "value": large_auroc - bean_raw,
            "flag": bool(np.isfinite(large_auroc - bean_raw) and large_auroc - bean_raw >= 0.05),
        },
        {
            "hypothesis": "crop presence is easier than mixture identity",
            "diagnostic": "best presence minus Bean-Maize AUROC",
            "value": best_presence - bean_raw,
            "flag": bool(np.isfinite(best_presence - bean_raw) and best_presence - bean_raw >= 0.05),
        },
    ]
    return pd.DataFrame(rows)


def build_gate_table(performance, deltas):
    task = "bean_maize_mixture"
    lens = "raw_safe"
    auroc = metric_value(performance, task, lens, "auroc")
    ap_lift = metric_value(performance, task, lens, "ap_lift")
    recall = metric_value(performance, task, lens, "recall")
    fpr = metric_value(performance, task, lens, "false_positive_rate")
    delta_lookup = {
        row.comparator: row.delta
        for row in deltas[deltas["task"].eq(task)].itertuples(index=False)
    }
    rows = [
        ("AUROC", auroc, ">= 0.75", np.isfinite(auroc) and auroc >= 0.75),
        ("AP lift", ap_lift, ">= 2.0", np.isfinite(ap_lift) and ap_lift >= 2.0),
        ("recall at calibrated rule", recall, ">= 0.60", np.isfinite(recall) and recall >= 0.60),
        ("held-out FPR", fpr, "<= 0.15", np.isfinite(fpr) and fpr <= 0.15),
        (
            "AUROC over nuisance",
            delta_lookup.get("nuisance_only", np.nan),
            ">= 0.05",
            np.isfinite(delta_lookup.get("nuisance_only", np.nan)) and delta_lookup["nuisance_only"] >= 0.05,
        ),
        (
            "AUROC over frozen TESSERA",
            delta_lookup.get("tessera_w2_frozen", np.nan),
            ">= 0.05",
            np.isfinite(delta_lookup.get("tessera_w2_frozen", np.nan)) and delta_lookup["tessera_w2_frozen"] >= 0.05,
        ),
    ]
    gate = pd.DataFrame(rows, columns=["criterion", "value", "rule", "passed"])
    gate["interpretation"] = "exploratory only; this snapshot has been repeatedly examined"
    return gate


def load_harvard_data(output_dir=OUTPUT_DIR, export_root=EXPORT_ROOT):
    require_inputs(output_dir)
    run = json.loads((output_dir / "run.json").read_text())
    run_fingerprint = str(run.get("run_fingerprint", "")).strip()
    if not run_fingerprint:
        raise RuntimeError(f"run_fingerprint is missing from {output_dir / 'run.json'}")
    export_dir = export_root / run_fingerprint[:16]
    figure_dir = export_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    fields = pd.read_parquet(output_dir / "fields.parquet")
    pixels = pd.read_parquet(output_dir / "pixels.parquet")
    memberships = pd.read_parquet(output_dir / "field_pixels.parquet")
    canonical_fields, canonical_memberships, attrition = canonical_harvard_cohort(
        fields, memberships
    )
    return {
        "output_dir": output_dir,
        "export_dir": export_dir,
        "figure_dir": figure_dir,
        "run": run,
        "run_fingerprint": run_fingerprint,
        "fields": fields,
        "pixels": pixels,
        "memberships": memberships,
        "canonical_fields": canonical_fields,
        "canonical_memberships": canonical_memberships,
        "attrition": attrition,
    }


def build_harvard_features(data_bundle):
    output_dir = data_bundle["output_dir"]
    export_dir = data_bundle["export_dir"]
    run_fingerprint = data_bundle["run_fingerprint"]
    canonical_fields = data_bundle["canonical_fields"]
    canonical_memberships = data_bundle["canonical_memberships"]
    attrition = data_bundle["attrition"].copy()

    pixel_features, embedding_pixels, cache_audit = extract_cached_features(
        output_dir, canonical_memberships, run_fingerprint
    )
    field_raw = aggregate_pixel_features(pixel_features)
    field_tessera = aggregate_embeddings(embedding_pixels)
    field_geometry = geometry_context(canonical_fields, canonical_memberships)
    base_columns = [
        "field_uid",
        "source_id",
        "landcover",
        "geometry_sha256",
        "utm_epsg",
        "area_m2",
        "bbox_width_m",
        "bbox_height_m",
        "pixel_count",
    ]
    field_index = (
        canonical_fields[base_columns]
        .merge(field_geometry, on="field_uid", how="inner", validate="one_to_one")
        .merge(field_raw, on="field_uid", how="inner", validate="one_to_one")
        .merge(field_tessera, on="field_uid", how="left", validate="one_to_one")
    )
    field_index = crop_targets(field_index)
    feature_sets = make_feature_sets(field_index)
    scoreable_fields = set(field_index["field_uid"])
    scoreable_memberships = canonical_memberships[
        canonical_memberships["field_uid"].isin(scoreable_fields)
    ].copy()
    attrition = pd.concat(
        [
            attrition,
            pd.DataFrame(
                [("raw_cache_scoreable_fields", len(field_index))],
                columns=["stage", "count"],
            ),
        ],
        ignore_index=True,
    )
    feature_contract = pd.DataFrame(
        [
            {
                "lens": lens,
                "feature_count": len(columns),
                "deployable_pre_survey": lens != "raw_full_retrospective",
                "role": (
                    "negative-control nuisance baseline"
                    if lens
                    in {"location_only", "geometry_acquisition_only", "nuisance_only"}
                    else "post-survey sensitivity only"
                    if lens == "raw_full_retrospective"
                    else "frozen representation baseline"
                    if lens == "tessera_w2_frozen"
                    else "candidate signal lens"
                ),
            }
            for lens, columns in feature_sets.items()
        ]
    )
    for filename, frame in (
        ("cohort_attrition.parquet", attrition),
        ("cache_task_audit.parquet", cache_audit),
        ("field_features.parquet", field_index),
        ("field_pixel_index.parquet", scoreable_memberships),
    ):
        write_frame_atomic(frame, export_dir / filename)
    return {
        **data_bundle,
        "canonical_memberships": scoreable_memberships,
        "attrition": attrition,
        "cache_audit": cache_audit,
        "field_index": field_index,
        "feature_sets": feature_sets,
        "feature_contract": feature_contract,
    }


def run_harvard_evaluation(feature_bundle):
    export_dir = feature_bundle["export_dir"]
    field_index = feature_bundle["field_index"]
    feature_sets = feature_bundle["feature_sets"]
    prediction_rows = []
    fold_rows = []
    audit_rows = []
    for task_name, spec in TASK_SPECS.items():
        task_frame = field_index[field_index["landcover"].isin(spec["labels"])].copy()
        task_frame = task_frame.sort_values("field_uid").reset_index(drop=True)
        folds, fold_status = spatial_folds(task_frame, spec["target"])
        audit_rows.append(
            {
                "task": task_name,
                "category": spec["category"],
                "scope": spec["scope"],
                "fields": len(task_frame),
                "positives": int(task_frame[spec["target"]].sum()),
                "spatial_blocks": task_frame["spatial_block"].nunique(),
                "fold_status": fold_status,
                "fold_count": len(folds),
            }
        )
        for fold_id, (_, test) in enumerate(folds):
            for index in test:
                fold_rows.append(
                    {
                        "task": task_name,
                        "field_uid": task_frame.iloc[index]["field_uid"],
                        "fold": fold_id,
                        "spatial_block": task_frame.iloc[index]["spatial_block"],
                    }
                )
        for lens, columns in feature_sets.items():
            if folds:
                prediction_rows.extend(
                    evaluate_lens(
                        task_frame,
                        spec["target"],
                        folds,
                        lens,
                        columns,
                        task_name,
                        spec,
                    )
                )
    predictions = pd.DataFrame(prediction_rows)
    if predictions.empty:
        raise RuntimeError(
            "No task supports leakage-safe spatial validation; inspect evaluation_audit"
        )
    fold_assignments = pd.DataFrame(fold_rows)
    evaluation_audit = pd.DataFrame(audit_rows)
    performance = summarize_performance(predictions)
    delta_rows = []
    for task_name in TASK_SPECS:
        for comparator in ("nuisance_only", "tessera_w2_frozen"):
            values = paired_delta(predictions, task_name, "raw_safe", comparator)
            delta_rows.append(
                {
                    "task": task_name,
                    "candidate": "raw_safe",
                    "comparator": comparator,
                    "metric": "auroc",
                    **values,
                }
            )
        values = paired_delta(
            predictions, task_name, "raw_full_retrospective", "raw_safe"
        )
        delta_rows.append(
            {
                "task": task_name,
                "candidate": "raw_full_retrospective",
                "comparator": "raw_safe",
                "metric": "auroc",
                **values,
            }
        )
    paired_deltas = pd.DataFrame(delta_rows)
    resolution = summarize_resolution(predictions)
    scorecard = build_scorecard(performance, paired_deltas, resolution)
    gate_table = build_gate_table(performance, paired_deltas)
    results = {
        "predictions": predictions,
        "fold_assignments": fold_assignments,
        "evaluation_audit": evaluation_audit,
        "performance": performance,
        "paired_deltas": paired_deltas,
        "resolution": resolution,
        "scorecard": scorecard,
        "gate_table": gate_table,
    }
    filenames = {
        "fold_assignments": "spatial_fold_assignments.parquet",
        "evaluation_audit": "evaluation_audit.parquet",
        "predictions": "field_predictions.parquet",
        "performance": "performance.parquet",
        "paired_deltas": "paired_lens_deltas.parquet",
        "resolution": "resolution_sensitivity.parquet",
        "scorecard": "hypothesis_scorecard.parquet",
        "gate_table": "bean_maize_exploratory_gates.parquet",
    }
    for key, filename in filenames.items():
        write_frame_atomic(results[key], export_dir / filename)
    return results


def record_figure(fig, feature_bundle, manifest, filename, description):
    path = feature_bundle["figure_dir"] / filename
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    manifest.append(
        {
            "figure": len(manifest) + 1,
            "filename": filename,
            "path": str(path),
            "description": description,
        }
    )
    return path


def figure_cohort_support(feature_bundle, manifest):
    field_index = feature_bundle["field_index"]
    cache_audit = feature_bundle["cache_audit"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    label_counts = field_index["landcover"].value_counts().reindex(TARGET_LABELS).fillna(0)
    axes[0].barh(
        label_counts.index,
        label_counts.values,
        color=[LABEL_COLORS[label] for label in label_counts.index],
    )
    axes[0].invert_yaxis()
    axes[0].set_title("Canonical Harvard fields")
    axes[0].set_xlabel("fields")
    resolution_counts = field_index["resolution_stratum"].value_counts().reindex(
        ["1-2 pixels", "3-9 pixels", "10+ pixels"]
    ).fillna(0)
    axes[1].bar(resolution_counts.index, resolution_counts.values, color="#0072b2")
    axes[1].set_title("10 m support per field")
    axes[1].set_ylabel("fields")
    axes[1].tick_params(axis="x", rotation=20)
    status_counts = cache_audit["status"].value_counts()
    axes[2].bar(status_counts.index, status_counts.values, color="#009e73")
    axes[2].set_title("Daily-cache recovery")
    axes[2].set_ylabel("tasks")
    axes[2].tick_params(axis="x", rotation=20)
    fig.suptitle("Figure 1 — cohort, spatial support, and cache provenance", fontsize=14)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "01_cohort_support_and_cache.png",
        "Canonical label counts, 10 m field support, and raw-cache task recovery.",
    )
    return fig, path


def figure_geography(feature_bundle, manifest):
    field_index = feature_bundle["field_index"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    for label in TARGET_LABELS:
        subset = field_index[field_index["landcover"].eq(label)]
        axes[0].scatter(
            subset["longitude"],
            subset["latitude"],
            s=18,
            alpha=0.7,
            color=LABEL_COLORS[label],
            label=label,
        )
    axes[0].set_title("Label geography (no external boundary data)")
    axes[0].set_xlabel("longitude")
    axes[0].set_ylabel("latitude")
    axes[0].legend(fontsize=8, frameon=False, loc="best")
    block_summary = (
        field_index.groupby("spatial_block")
        .agg(fields=("field_uid", "size"), intercrop_rate=("is_intercrop", "mean"))
        .reset_index()
    )
    axes[1].scatter(
        np.arange(len(block_summary)),
        block_summary["intercrop_rate"],
        s=15 + 5 * np.sqrt(block_summary["fields"]),
        c=block_summary["intercrop_rate"],
        cmap="viridis",
        vmin=0,
        vmax=1,
    )
    axes[1].axhline(field_index["is_intercrop"].mean(), color="black", ls="--", lw=1)
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].set_title(f"Intercrop prevalence across {BLOCK_METRES // 1000} km blocks")
    axes[1].set_xlabel("spatial block (ordered only for display)")
    axes[1].set_ylabel("intercrop fraction")
    fig.suptitle("Figure 2 — geography is a competing explanation", fontsize=14)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "02_geography_and_block_prevalence.png",
        "Harvard locations and block prevalence used to diagnose geographic confounding.",
    )
    return fig, path


def figure_multilens_performance(feature_bundle, results, manifest):
    performance = results["performance"]
    tasks = ["generic_intercrop", "bean_maize_mixture", "potato_maize_mixture"]
    lenses = list(feature_bundle["feature_sets"])
    matrix = (
        performance[performance["task"].isin(tasks)]
        .pivot(index="task", columns="lens", values="auroc")
        .reindex(index=tasks, columns=lenses)
    )
    fig, ax = plt.subplots(figsize=(14, 4.8))
    image = ax.imshow(matrix.to_numpy(dtype=float), vmin=0.4, vmax=0.9, cmap="RdYlBu")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            label = "—" if not np.isfinite(value) else f"{value:.2f}"
            ax.text(column, row, label, ha="center", va="center", fontsize=9)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title("Figure 3 — spatially held-out AUROC across competing lenses")
    fig.colorbar(image, ax=ax, label="AUROC")
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "03_multilens_intercropping_performance.png",
        "Spatially held-out AUROC across nuisance, raw, retrospective, and TESSERA lenses.",
    )
    return fig, path


def figure_crop_presence(feature_bundle, results, manifest):
    performance = results["performance"]
    presence = performance[performance["task_category"].eq("crop_presence")]
    tasks = list(TASK_SPECS)[3:]
    lenses = ["nuisance_only", "raw_safe", "tessera_w2_frozen"]
    fig, ax = plt.subplots(figsize=(11, 5.2))
    x = np.arange(len(tasks))
    width = 0.24
    for offset, lens in enumerate(lenses):
        values = presence[presence["lens"].eq(lens)].set_index("task")["auroc"].reindex(tasks)
        ax.bar(
            x + (offset - 1) * width,
            values,
            width=width,
            label=lens,
            color=LENS_COLORS[lens],
        )
    ax.axhline(0.5, color="black", lw=1, ls="--")
    ax.set_ylim(0.35, 1.0)
    ax.set_xticks(x, [task.replace("_presence", "") for task in tasks])
    ax.set_ylabel("spatially held-out AUROC")
    ax.set_title("Figure 4 — crop presence asks a different question")
    ax.legend(frameon=False)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "04_crop_presence_heads.png",
        "Crop-presence AUROC compared with nuisance and frozen baselines.",
    )
    return fig, path


def figure_resolution(feature_bundle, results, manifest):
    resolution = results["resolution"]
    tasks = ["generic_intercrop", "bean_maize_mixture"]
    strata = ["1-2 pixels", "3-9 pixels", "10+ pixels"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, task in zip(axes, tasks):
        subset = resolution[resolution["task"].eq(task)].set_index("resolution_stratum")
        values = subset["auroc"].reindex(strata)
        counts = subset["n"].reindex(strata)
        bars = ax.bar(strata, values, color=["#d9d9d9", "#56b4e9", "#0072b2"])
        for bar, count in zip(bars, counts):
            if np.isfinite(count):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    0.38,
                    f"n={int(count)}",
                    rotation=90,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )
        ax.axhline(0.5, color="black", ls="--", lw=1)
        ax.set_ylim(0.35, 1.0)
        ax.set_title(task)
        ax.tick_params(axis="x", rotation=20)
    axes[0].set_ylabel("raw-safe AUROC")
    fig.suptitle("Figure 5 — separability by field support", fontsize=14)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "05_resolution_sensitivity.png",
        "Raw-safe performance by private 10 m pixel count.",
    )
    return fig, path


def figure_phenology(feature_bundle, manifest):
    field_index = feature_bundle["field_index"]
    families = [
        ("Bean–Maize family", ["Bean", "Maize", "Bean and Maize"]),
        ("Potato–Maize family", ["Irish Potato", "Maize", "Irish Potato and Maize"]),
    ]
    phase_columns = {phase: f"{phase}_s2_ndvi_med__fmed" for phase in PHASES}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    for ax, (title, labels) in zip(axes, families):
        for label in labels:
            subset = field_index[field_index["landcover"].eq(label)]
            values = [subset[column].median() for column in phase_columns.values()]
            ax.plot(
                list(phase_columns),
                values,
                marker="o",
                lw=2,
                color=LABEL_COLORS[label],
                label=f"{label} (n={len(subset)})",
            )
        ax.axvline(1.5, color="black", ls="--", lw=1)
        ax.text(
            1.45,
            ax.get_ylim()[1],
            "survey start",
            rotation=90,
            va="top",
            ha="right",
            fontsize=8,
        )
        ax.set_title(title)
        ax.set_xlabel("2025 phase")
        ax.legend(frameon=False, fontsize=8)
    axes[0].set_ylabel("field median NDVI")
    fig.suptitle("Figure 6 — pre-survey and retrospective phenology", fontsize=14)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "06_safe_and_retrospective_phenology.png",
        "March-April pre-survey and May-June retrospective NDVI trajectories.",
    )
    return fig, path


def figure_falsification(feature_bundle, results, manifest):
    deltas = results["paired_deltas"]
    scorecard = results["scorecard"]
    gate_table = results["gate_table"]
    bean_deltas = deltas[deltas["task"].eq("bean_maize_mixture")].copy()
    labels = [
        f"{row.candidate}\nminus {row.comparator}"
        for row in bean_deltas.itertuples(index=False)
    ]
    values = bean_deltas["delta"].to_numpy(dtype=float)
    low = bean_deltas["ci_low"].to_numpy(dtype=float)
    high = bean_deltas["ci_high"].to_numpy(dtype=float)
    errors = np.maximum(0, np.vstack([values - low, high - values]))
    errors[~np.isfinite(errors)] = 0
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
    axes[0].barh(
        np.arange(len(values)),
        values,
        xerr=errors,
        color=["#0072b2", "#cc79a7", "#d55e00"][: len(values)],
        alpha=0.85,
    )
    axes[0].axvline(0, color="black", lw=1)
    axes[0].set_yticks(np.arange(len(labels)), labels)
    axes[0].set_xlabel("paired AUROC difference (block bootstrap 95% interval)")
    axes[0].set_title("Bean–Maize lens comparisons")
    axes[1].axis("off")
    lines = ["Competing explanations"]
    for row in scorecard.itertuples(index=False):
        symbol = "FLAG" if row.flag else "not flagged"
        value = "NA" if not np.isfinite(row.value) else f"{row.value:+.3f}"
        lines.append(f"{symbol:11s}  {value:>7s}  {row.hypothesis}")
    lines.extend(["", "Predeclared exploratory Bean–Maize gates"])
    for row in gate_table.itertuples(index=False):
        symbol = "PASS" if row.passed else "FAIL"
        value = "NA" if not np.isfinite(row.value) else f"{row.value:.3f}"
        lines.append(f"{symbol:4s}  {value:>6s}  {row.criterion} ({row.rule})")
    axes[1].text(
        0.01,
        0.98,
        "\n".join(lines),
        va="top",
        ha="left",
        family="monospace",
        fontsize=9,
    )
    axes[1].set_title("What the Harvard-only evidence supports")
    fig.suptitle("Figure 7 — falsification scorecard", fontsize=14)
    fig.tight_layout()
    path = record_figure(
        fig,
        feature_bundle,
        manifest,
        "07_falsification_scorecard.png",
        "Paired lens deltas, competing explanations, and exploratory gates.",
    )
    return fig, path


def finalize_harvard_workbench(data_bundle, feature_bundle, results, figure_manifest):
    export_dir = data_bundle["export_dir"]
    run_fingerprint = data_bundle["run_fingerprint"]
    field_index = feature_bundle["field_index"]
    memberships = feature_bundle["canonical_memberships"]
    performance = results["performance"]
    if len(figure_manifest) != 7:
        raise RuntimeError(f"expected seven figures, found {len(figure_manifest)}")
    write_frame_atomic(pd.DataFrame(figure_manifest), export_dir / "figure_manifest.parquet")
    primary_rows = performance[
        performance["task"].isin(
            ["generic_intercrop", "bean_maize_mixture", "potato_maize_mixture"]
        )
    ][
        [
            "task",
            "lens",
            "n",
            "positive_n",
            "auroc",
            "auroc_ci_low",
            "auroc_ci_high",
            "average_precision",
            "ap_lift",
            "recall",
            "false_positive_rate",
        ]
    ]
    handoff = {
        "analysis": "Harvard-only self-contained workbench v1",
        "run_fingerprint": run_fingerprint,
        "export_dir": str(export_dir),
        "source_contract": {
            "labels_and_geometry": "reduced Harvard parquet frozen in the v2 run",
            "imagery": "daily S1/S2 arrays already retained in the v2 task caches",
            "external_data_or_requests": False,
            "new_tessera_inference": False,
            "safe_interval": f"[{SAFE_START.date()}, {SAFE_END.date()})",
            "retrospective_interval": f"[{SAFE_END.date()}, {FULL_END.date()})",
        },
        "cohort": {
            "canonical_scoreable_fields": len(field_index),
            "private_pixels": len(memberships),
            "spatial_blocks": field_index["spatial_block"].nunique(),
            "label_counts": field_index["landcover"].value_counts().to_dict(),
        },
        "validation": {
            "outer_split": f"StratifiedGroupKFold on {BLOCK_METRES // 1000} km blocks; no random fallback",
            "primary_metrics": ["AUROC", "average precision", "AP lift"],
            "operating_rule": f"inner spatial calibration at target FPR {TARGET_TRAIN_FPR:.0%}; no refit",
            "uncertainty": f"{BOOTSTRAP_REPLICATES} spatial-block bootstrap replicates",
            "status": "exploratory; the snapshot has been repeatedly examined",
        },
        "performance": primary_rows.to_dict(orient="records"),
        "paired_deltas": results["paired_deltas"].to_dict(orient="records"),
        "scorecard": results["scorecard"].to_dict(orient="records"),
        "bean_maize_gates": results["gate_table"].to_dict(orient="records"),
        "figures": figure_manifest,
        "limitations": [
            "The reduced parquet omits per-field planting date, visit date, crop stage, management, and photographs.",
            "A common Season B calendar is used; it is not a field-specific phenology clock.",
            "Survey prevalence must not be extrapolated to Rwanda.",
            "Ten-metre center-selected pixels can contain boundary or neighboring-land signal.",
            "No result estimates crop fraction, plant count, biomass, yield, or causal benefit.",
        ],
    }
    write_json_atomic(export_dir / "HARVARD_WORKBENCH_HANDOFF.json", handoff)
    required_exports = [
        "cohort_attrition.parquet",
        "cache_task_audit.parquet",
        "field_features.parquet",
        "field_pixel_index.parquet",
        "spatial_fold_assignments.parquet",
        "evaluation_audit.parquet",
        "field_predictions.parquet",
        "performance.parquet",
        "paired_lens_deltas.parquet",
        "resolution_sensitivity.parquet",
        "hypothesis_scorecard.parquet",
        "bean_maize_exploratory_gates.parquet",
        "figure_manifest.parquet",
        "HARVARD_WORKBENCH_HANDOFF.json",
    ]
    missing = [name for name in required_exports if not (export_dir / name).is_file()]
    if missing:
        raise RuntimeError(f"incomplete export: {missing}")
    completion = {
        "status": "complete",
        "run_fingerprint": run_fingerprint,
        "export_dir": str(export_dir),
        "field_count": len(field_index),
        "prediction_count": len(results["predictions"]),
        "figure_count": len(figure_manifest),
        "required_exports": required_exports,
    }
    write_json_atomic(export_dir / "COMPLETED.json", completion)
    print("HARVARD_WORKBENCH_HANDOFF_BEGIN")
    print(json.dumps(_jsonable(handoff), indent=2, sort_keys=True))
    print("HARVARD_WORKBENCH_HANDOFF_END")
    print(f"Completed Harvard-only workbench: {export_dir}")
    return handoff

# Execute the complete workbench in one scope.
data_bundle = load_harvard_data()
display(data_bundle["attrition"])
print(
    f"Frozen run: {data_bundle['run_fingerprint'][:16]} | "
    f"canonical fields: {len(data_bundle['canonical_fields']):,} | "
    f"private pixels: {len(data_bundle['canonical_memberships']):,}"
)

feature_bundle = build_harvard_features(data_bundle)
display(feature_bundle["feature_contract"])
display(
    feature_bundle["cache_audit"]["status"]
    .value_counts(dropna=False)
    .rename("tasks")
    .to_frame()
)

results = run_harvard_evaluation(feature_bundle)
display(results["evaluation_audit"])
display(
    results["performance"]
    .loc[lambda frame: frame["task_category"].eq("intercropping")]
    [["task", "lens", "n", "positive_n", "auroc", "average_precision", "ap_lift", "recall", "false_positive_rate"]]
)

figure_manifest = []
figure_1, figure_1_path = figure_cohort_support(feature_bundle, figure_manifest)
display(figure_1)

figure_2, figure_2_path = figure_geography(feature_bundle, figure_manifest)
display(figure_2)

figure_3, figure_3_path = figure_multilens_performance(
    feature_bundle, results, figure_manifest
)
display(figure_3)

figure_4, figure_4_path = figure_crop_presence(
    feature_bundle, results, figure_manifest
)
display(figure_4)

figure_5, figure_5_path = figure_resolution(
    feature_bundle, results, figure_manifest
)
display(figure_5)

figure_6, figure_6_path = figure_phenology(feature_bundle, figure_manifest)
display(figure_6)

figure_7, figure_7_path = figure_falsification(
    feature_bundle, results, figure_manifest
)
display(figure_7)

handoff = finalize_harvard_workbench(
    data_bundle,
    feature_bundle,
    results,
    figure_manifest,
)